# ETL: Raw → Silver Layer

Este notebook realiza o processo completo de ETL (Extract, Transform, Load) dos dados brutos (camada Raw) para a camada Silver (dados tratados e normalizados) do Data Lake.

## Processo:
1. **Extract**: Carrega dados brutos dos arquivos CSV (2024 e 2025)
2. **Transform**: Aplica transformações, limpeza e normalização
3. **Load**: Carrega dados tratados no PostgreSQL (schema `dl`)

---

## 1. Importações e Configuração

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

### Configuração de Conexão com Banco de Dados

In [2]:
# Configuração do banco de dados PostgreSQL
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "sinistros_prf")
DB_USER = os.getenv("DB_USER", "prf_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "prf_pass")

# String de conexão
CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Criar engine SQLAlchemy
engine = create_engine(CONNECTION_STRING, pool_size=10, max_overflow=20, echo=False)

print("Configuração do banco definida")
print(f"Host: {DB_HOST}:{DB_PORT}")
print(f"Database: {DB_NAME}")

Configuração do banco definida
Host: localhost:5432
Database: sinistros_prf


### Funções de Transformação de Dados

In [3]:
# Mapeamentos para transformações
UF_TO_LOCALIDADE = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas",
    "BA": "Bahia", "CE": "Ceará", "DF": "Distrito Federal",
    "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão",
    "MT": "Mato Grosso", "MS": "Mato Grosso do Sul", "MG": "Minas Gerais",
    "PA": "Pará", "PB": "Paraíba", "PR": "Paraná", "PE": "Pernambuco",
    "PI": "Piauí", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul", "RO": "Rondônia", "RR": "Roraima",
    "SC": "Santa Catarina", "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins"
}

UF_TO_REGIAO = {
    "AC": "Norte", "AL": "Nordeste", "AP": "Norte", "AM": "Norte",
    "BA": "Nordeste", "CE": "Nordeste", "DF": "Centro-Oeste",
    "ES": "Sudeste", "GO": "Centro-Oeste", "MA": "Nordeste",
    "MT": "Centro-Oeste", "MS": "Centro-Oeste", "MG": "Sudeste",
    "PA": "Norte", "PB": "Nordeste", "PR": "Sul", "PE": "Nordeste",
    "PI": "Nordeste", "RJ": "Sudeste", "RN": "Nordeste", "RS": "Sul",
    "RO": "Norte", "RR": "Norte", "SC": "Sul", "SP": "Sudeste",
    "SE": "Nordeste", "TO": "Norte"
}

DIAS_SEMANA = {
    0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira",
    3: "Quinta-feira", 4: "Sexta-feira", 5: "Sábado", 6: "Domingo"
}

def transformar_dados_silver(df):
    """
    Pipeline completo de transformação para a camada Silver.
    Aplica todas as transformações necessárias nos dados brutos.
    """
    print("\nINICIANDO TRANSFORMAÇÃO SILVER")
    print(f"Shape original: {df.shape}\n")
    
    # 1. Normalização de strings
    print("Normalizando strings...")
    def normalize(s):
        s_str = s.astype(str).str.strip()
        s = s_str.replace({
            "NaN": "", "None": "", "NoneType": "", "(null)": "",
            "na": "", "n/a": "", "N/A": "", "NULL": "", "null": "", "nan": ""
        })
        s = s.str.replace(",", ".", regex=False)
        s = s.replace("", pd.NA)
        return s
    
    for col in df.columns:
        df[col] = normalize(df[col])
    
    # 2. Conversão de tipos
    print("Convertendo tipos de dados...")
    int_cols = ["id", "pesid", "id_veiculo", "idade", "ano_fabricacao_veiculo",
                "ordem_tipo_acidente", "br", "ilesos", "feridos_leves", 
                "feridos_graves", "mortos"]
    
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(pd.Int64Dtype())
    
    for col in ["km", "latitude", "longitude"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(pd.Float64Dtype())
    
    str_cols = ["uf", "municipio", "causa_principal", "causa_acidente",
                "tipo_acidente", "sentido_via", "condicao_metereologica",
                "tipo_pista", "tracado_via", "uso_solo", "tipo_veiculo",
                "marca", "estado_fisico", "sexo", "tipo_envolvido"]
    
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(pd.StringDtype())
    
    if "data_inversa" in df.columns:
        df["data_inversa"] = pd.to_datetime(df["data_inversa"], format="%Y-%m-%d", errors='coerce')
    
    if "horario" in df.columns:
        parsed = pd.to_datetime(df["horario"].astype(str), format="%H:%M:%S", errors='coerce')
        df["horario"] = parsed.dt.time
    
    # 3. Renomear colunas
    print("Renomeando colunas...")
    rename_map = {
        "ano_fabricacao_veiculo": "veiculo_ano_fabricacao", "br": "rodovia",
        "causa_acidente": "sinistro_causa", "causa_principal": "sinistro_causa_principal",
        "condicao_metereologica": "condicao_meteorologica", "data_inversa": "data",
        "id_veiculo": "veiculo_id", "id": "sinistro_id", "idade": "envolvido_idade",
        "km": "quilometro", "marca": "veiculo_marca_modelo", "pesid": "id_envolvido",
        "ordem_tipo_acidente": "sinistro_ordem_tipo", "sentido_via": "via_sentido",
        "sexo": "envolvido_sexo", "tipo_acidente": "sinistro_tipo",
        "tipo_envolvido": "envolvido_tipo", "tipo_pista": "via_tipo",
        "tipo_veiculo": "veiculo_tipo", "tracado_via": "via_tracado"
    }
    
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    df.rename(columns=existing, inplace=True)
    
    # 4. Criar colunas derivadas
    print("Criando colunas derivadas...")
    
    # Feridos total
    if all(col in df.columns for col in ["feridos_leves", "feridos_graves"]):
        df["feridos"] = (
            df["feridos_leves"].fillna(0).astype(pd.Int64Dtype()) + 
            df["feridos_graves"].fillna(0).astype(pd.Int64Dtype())
        )
    
    # Rodovia formatada
    if "rodovia" in df.columns:
        df["rodovia"] = df["rodovia"].apply(
            lambda v: f"BR-{str(v).zfill(3)}" if pd.notna(v) else pd.NA
        ).astype(pd.StringDtype())
        df["rodovia_numero"] = df["rodovia"].str.split('-').str[-1]
    
    # Data/Hora
    if all(col in df.columns for col in ["data", "horario"]):
        df["data_hora"] = pd.to_datetime(
            df["data"].astype(str) + " " + df["horario"].astype(str),
            errors='coerce'
        )
        df["dia_semana"] = df["data_hora"].dt.weekday.map(DIAS_SEMANA)
        df["data"] = df["data_hora"].dt.date
        df["ano"] = df["data_hora"].dt.year.astype(pd.Int64Dtype())
        df["hora"] = df["data_hora"].dt.hour.astype(pd.Int64Dtype())
    
    # Período do dia
    if "hora" in df.columns:
        def period_of_day(h):
            if pd.isna(h): return pd.NA
            h = int(h)
            if 0 <= h <= 5: return "Madrugada"
            if 6 <= h <= 11: return "Manhã"
            if 12 <= h <= 17: return "Tarde"
            return "Noite"
        
        df["periodo"] = df["hora"].apply(period_of_day).astype(pd.StringDtype())
    
    # Período da semana
    if "dia_semana" in df.columns:
        df["periodo_semana"] = df["dia_semana"].apply(
            lambda d: "Final de semana" if d in ["Domingo", "Sábado"] 
            else "Segunda à Sexta" if pd.notna(d) else pd.NA
        ).astype(pd.StringDtype())
    
    # Localidade e Região
    if "uf" in df.columns:
        df["localidade"] = df["uf"].map(UF_TO_LOCALIDADE).astype(pd.StringDtype())
        df["regiao"] = df["uf"].map(UF_TO_REGIAO).astype(pd.StringDtype())
    
    # Faixas etárias
    if "envolvido_idade" in df.columns:
        def idade_to_faixa(idade):
            if pd.isna(idade) or idade < 0: return pd.NA
            idade = int(idade)
            if idade <= 9: return "0-9"
            elif idade <= 19: return "10-19"
            elif idade <= 29: return "20-29"
            elif idade <= 39: return "30-39"
            elif idade <= 49: return "40-49"
            elif idade <= 59: return "50-59"
            elif idade <= 69: return "60-69"
            elif idade <= 79: return "70-79"
            elif idade <= 89: return "80-89"
            elif idade <= 99: return "90-99"
            return "100+"
        
        df["faixa_etaria_ano"] = df["envolvido_idade"].apply(idade_to_faixa).astype(pd.StringDtype())
        
        def idade_to_classe(idade):
            if pd.isna(idade) or idade < 0: return pd.NA
            idade = int(idade)
            if idade <= 11: return "Criança"
            if 12 <= idade <= 17: return "Adolescente"
            if 18 <= idade <= 59: return "Adulto"
            return "Idoso"
        
        df["faixa_etaria_classe"] = df["envolvido_idade"].apply(idade_to_classe).astype(pd.StringDtype())
    
    # Gravidade
    if all(col in df.columns for col in ["mortos", "feridos"]):
        mortos = pd.to_numeric(df["mortos"], errors='coerce').fillna(0)
        feridos = pd.to_numeric(df["feridos"], errors='coerce').fillna(0)
        
        df["gravidade"] = np.select(
            [mortos > 0, feridos > 0, feridos == 0],
            ["Com morto", "Com ferido", "Sem vítima"],
            default="Não informado"
        )
        df["gravidade"] = df["gravidade"].astype(pd.StringDtype())
    
    # UPS (Unidade Padrão de Severidade)
    if all(col in df.columns for col in ["mortos", "feridos", "sinistro_tipo"]):
        mortos = pd.to_numeric(df["mortos"], errors='coerce').fillna(0)
        feridos = pd.to_numeric(df["feridos"], errors='coerce').fillna(0)
        tipo = df["sinistro_tipo"].fillna("")
        
        ups_values = np.where(
            mortos > 0, 13,
            np.where(tipo.str.contains("Atropelamento", na=False), 6,
                    np.where(feridos > 0, 4, 1))
        )
        df["ups"] = pd.array(ups_values, dtype=pd.Int64Dtype())
    
    # De-Para
    if "uso_solo" in df.columns:
        df["uso_solo"] = df["uso_solo"].replace({"Sim": "Urbano", "Não": "Rural"})
    
    # 5. Limpeza de dados
    print("Limpando dados...")
    
    # Tratamento de outliers
    if "envolvido_idade" in df.columns:
        df.loc[df["envolvido_idade"] > 200, "envolvido_idade"] = pd.NA
    
    if "veiculo_ano_fabricacao" in df.columns:
        ano_atual = datetime.now().year
        df.loc[(df["veiculo_ano_fabricacao"] > ano_atual) | 
               (df["veiculo_ano_fabricacao"] < 1920), "veiculo_ano_fabricacao"] = pd.NA
    
    # Remover colunas desnecessárias
    cols_to_drop = [
        c
        for c in [
            "regional",
            "uop",
            "delegacia",
            "classificacao_acidente",
            "fase_dia",
        ]
        if c in df.columns
    ]

    if cols_to_drop:
        df.drop(cols_to_drop, axis=1, inplace=True)
    
    # Remover duplicatas
    before = len(df)
    df = df.drop_duplicates()
    if before != len(df):
        print(f"Removidas {before - len(df)} duplicatas")
    
    print(f"\nShape final: {df.shape}")
    return df


print("Funções de transformação definidas")

Funções de transformação definidas


## 2. Extract - Carregamento dos Dados Brutos

In [4]:
# Caminhos dos arquivos brutos
file_2024 = '../data_layer/raw/acidentes2024_todas_causas_tipos.csv'
file_2025 = '../data_layer/raw/acidentes2025_todas_causas_tipos.csv'

print("Carregando dados brutos...")
print(f"Arquivo 2024: {file_2024}")
print(f"Arquivo 2025: {file_2025}")

# Carregar dados
df_2024 = pd.read_csv(file_2024, sep=',', encoding='utf-8', low_memory=False)
df_2025 = pd.read_csv(file_2025, sep=',', encoding='utf-8', low_memory=False)

print(f"\nRegistros 2024: {len(df_2024):,}")
print(f"Registros 2025: {len(df_2025):,}")

# Concatenar os dados
df_raw = pd.concat([df_2024, df_2025], ignore_index=True)

print(f"\nTotal de registros brutos: {len(df_raw):,}")
print(f"Colunas: {len(df_raw.columns)}")

Carregando dados brutos...
Arquivo 2024: ../data_layer/raw/acidentes2024_todas_causas_tipos.csv
Arquivo 2025: ../data_layer/raw/acidentes2025_todas_causas_tipos.csv

Registros 2024: 603,215
Registros 2025: 378,575

Total de registros brutos: 981,790
Colunas: 37

Registros 2024: 603,215
Registros 2025: 378,575

Total de registros brutos: 981,790
Colunas: 37


## 3. Transform - Transformação e Limpeza dos Dados

Aplicando todas as transformações necessárias para converter dados brutos em dados normalizados da camada Silver.

In [5]:
# Aplicar todas as transformações
df_silver = transformar_dados_silver(df_raw.copy())

print(f"\nEstatísticas da transformação:")
print(f"Registros transformados: {len(df_silver):,}")
print(f"Colunas finais: {len(df_silver.columns)}")
if 'sinistro_id' in df_silver.columns:
    print(f"Sinistros únicos: {df_silver['sinistro_id'].nunique():,}")
if 'id_envolvido' in df_silver.columns:
    print(f"Pessoas envolvidas: {df_silver['id_envolvido'].nunique():,}")
if 'veiculo_id' in df_silver.columns:
    print(f"Veículos envolvidos: {df_silver['veiculo_id'].nunique():,}")


INICIANDO TRANSFORMAÇÃO SILVER
Shape original: (981790, 37)

Normalizando strings...
Convertendo tipos de dados...
Convertendo tipos de dados...
Renomeando colunas...
Criando colunas derivadas...
Renomeando colunas...
Criando colunas derivadas...
Limpando dados...
Limpando dados...

Shape final: (981790, 45)

Estatísticas da transformação:
Registros transformados: 981,790
Colunas finais: 45
Sinistros únicos: 120,348
Pessoas envolvidas: 294,539
Veículos envolvidos: 230,691

Shape final: (981790, 45)

Estatísticas da transformação:
Registros transformados: 981,790
Colunas finais: 45
Sinistros únicos: 120,348
Pessoas envolvidas: 294,539
Veículos envolvidos: 230,691


## 4. Load - Carregamento no PostgreSQL

Carregar os dados transformados no banco de dados PostgreSQL (schema `silver`, tabela `tb_sinistros_silver`).

In [6]:
# Testar conexão com banco
print("Testando conexão com PostgreSQL...")
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Conexão estabelecida com sucesso!")
except Exception as e:
    print(f"Erro na conexão: {e}")
    print("Verifique se o Docker está rodando e se as credenciais estão corretas.")

Testando conexão com PostgreSQL...
Conexão estabelecida com sucesso!


In [7]:
# Carregar dados no PostgreSQL
print("\nCarregando dados no PostgreSQL...")
print(f"Schema: dl")
print(f"Tabela: tb_sinistros_silver")
print(f"Registros: {len(df_silver):,}\n")

try:
    df_silver.to_sql(
        'tb_sinistros_silver', 
        engine, 
        schema='dl',
        if_exists='replace',
        index=False,
        method='multi',
        chunksize=1000
    )
    print("Dados carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar dados: {e}")


Carregando dados no PostgreSQL...
Schema: dl
Tabela: tb_sinistros_silver
Registros: 981,790

Dados carregados com sucesso!
Dados carregados com sucesso!


In [8]:
# Verificar dados carregados
print("\nVerificando dados no banco...")
query = "SELECT COUNT(*) as total FROM dl.tb_sinistros_silver"
result = pd.read_sql(query, engine)
print(f"Total de registros no banco: {result['total'][0]:,}")

# Consulta de amostra
print("\nPrimeiras linhas no banco:\n")
query_sample = "SELECT * FROM dl.tb_sinistros_silver LIMIT 5"
pd.read_sql(query_sample, engine)


Verificando dados no banco...
Total de registros no banco: 981,790

Primeiras linhas no banco:



,sinistro_id,id_envolvido,data,dia_semana,horario,uf,rodovia,quilometro,municipio,sinistro_causa_principal,...,ano,hora,periodo,periodo_semana,localidade,regiao,faixa_etaria_ano,faixa_etaria_classe,gravidade,ups
0,571772,1268971,2024-01-01,Segunda-feira,00:05:00,RJ,BR-101.0,272.5,TANGUA,Sim,...,2024,0,Madrugada,Segunda à Sexta,Rio de Janeiro,Sudeste,20-29,Adulto,Com morto,13
1,571774,1268985,2024-01-01,Segunda-feira,00:05:00,GO,BR-153.0,424.6,ANAPOLIS,Não,...,2024,0,Madrugada,Segunda à Sexta,Goiás,Centro-Oeste,30-39,Adulto,Sem vítima,1
2,571774,1268985,2024-01-01,Segunda-feira,00:05:00,GO,BR-153.0,424.6,ANAPOLIS,Sim,...,2024,0,Madrugada,Segunda à Sexta,Goiás,Centro-Oeste,30-39,Adulto,Sem vítima,1
3,571777,1269020,2024-01-01,Segunda-feira,01:45:00,ES,BR-101.0,264.1,SERRA,Sim,...,2024,1,Madrugada,Segunda à Sexta,Espírito Santo,Sudeste,50-59,Adulto,Sem vítima,1
4,571778,1269028,2024-01-01,Segunda-feira,00:45:00,SC,BR-101.0,110.0,PENHA,Não,...,2024,0,Madrugada,Segunda à Sexta,Santa Catarina,Sul,50-59,Adulto,Sem vítima,1


## 5. Resumo Final

ETL Raw → Silver concluído com sucesso!

**Próximos passos:**
1. Execute o notebook `etl_silver_to_gold.ipynb` para criar o Data Warehouse dimensional
2. Explore os dados na camada Silver usando `data_layer/silver/analytics.ipynb`